In [ ]:
import pandas as pd

# Path to the CSV file containing bounding box annotations
csv_path = 'C:/Users/samya/PyCharmProject/Pneumonia-Detection_dataset/data/stage_2_train_labels.csv'

# Load the full dataset
labels_df = pd.read_csv(csv_path)

# Filter to include only rows where pneumonia is present (Target == 1)
pneumonia_df = labels_df[labels_df['Target'] == 1].copy()

#reset index for cleaner downstream processing
pneumonia_df.reset_index(drop=True, inplace=True)

In [1]:
import numpy as np

def create_segmentation_mask(img_id, boxes_df, orig_size=(1024, 1024), new_size=(64, 64)):
    """
    Generate a binary segmentation mask from bounding boxes for a specific image.

    Parameters:
        img_id (str or int): Identifier for the image (matches 'patientId' in boxes_df).
        boxes_df (pd.DataFrame): DataFrame containing bounding box annotations with columns:
                                 ['patientId', 'x', 'y', 'width', 'height'].
        orig_size (tuple): Original image dimensions as (width, height).
        new_size (tuple): Desired output mask size as (width, height).

    Returns:
        np.ndarray: A binary mask of shape `new_size` with 1s inside bounding boxes.
    """
    # Initialize empty mask
    mask = np.zeros(new_size, dtype=np.uint8)

    # Filter bounding boxes for the given image ID
    boxes = boxes_df[boxes_df['patientId'] == img_id]

    # Compute scaling factors
    scale_x = new_size[0] / orig_size[0]
    scale_y = new_size[1] / orig_size[1]

    for _, row in boxes.iterrows():
        # Scale and round bounding box coordinates
        x1 = int(row['x'] * scale_x)
        y1 = int(row['y'] * scale_y)
        x2 = int((row['x'] + row['width']) * scale_x)
        y2 = int((row['y'] + row['height']) * scale_y)

        # Clip coordinates to mask boundaries
        x1, x2 = np.clip([x1, x2], 0, new_size[0])
        y1, y2 = np.clip([y1, y2], 0, new_size[1])

        # Fill mask region
        mask[y1:y2, x1:x2] = 1

    return mask

In [ ]:
import os
import pydicom
import cv2
import numpy as np
from tqdm import tqdm

TRAIN_IMG_DIR = "C:/Users/samya/PyCharmProject/Pneumonia-Detection_dataset/data/stage_2_train_images"

image_paths = [os.path.join(TRAIN_IMG_DIR, f) for f in os.listdir(TRAIN_IMG_DIR) if f.endswith(".dcm")]

X = []
y = []

for path in tqdm(image_paths):  # Use a subset for faster development
    img_id = os.path.basename(path).replace('.dcm', '')
    ds = pydicom.dcmread(path)
    img = ds.pixel_array
    img = cv2.resize(img, (64, 64))
    img = img / 255.0
    img_rgb = np.repeat(img[..., np.newaxis], 3, axis=-1)

    mask = create_segmentation_mask(img_id, labels_df, orig_size=ds.pixel_array.shape, new_size=(64, 64))
    mask = mask[..., np.newaxis]  # shape: (64, 64, 1)

    X.append(img_rgb)
    y.append(mask)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.uint8)


In [ ]:
import tensorflow as tf

# Clear, Flexible Exponential Decay Function
def exponential_decay(lr_initial, decay_steps, decay_rate=0.1):
    """
    Returns a function that computes exponentially decaying learning rate.
    
    Parameters:
    - lr_initial: Initial learning rate
    - decay_steps: Controls the rate of decay
    - decay_rate: The base of the exponential decay (default: 0.1)

    Returns:
    - A function that takes an epoch index and returns the decayed learning rate
    """
    def schedule(epoch):
        return lr_initial * decay_rate ** (epoch / decay_steps)
    return schedule

# Define the scheduler function
exponential_decay_fn = exponential_decay(lr_initial=0.01, decay_steps=20)

# Learning Rate Scheduler Callback
lr_scheduler_cb = tf.keras.callbacks.LearningRateScheduler(exponential_decay_fn, verbose=1)

# Model Checkpoint Callback (saves best model only)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath="xray_model.h5",
    save_best_only=True,
    monitor="val_loss",
    mode="min",
    verbose=1
)

# Early Stopping Callback (restores best weights after patience period)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True,
    monitor="val_loss",
    mode="min",
    verbose=1
)


In [ ]:
# These our our scoring metrics that are going to be used to evaluate our models
METRICS = ['accuracy', 
           tf.keras.metrics.Precision(name='precision'), 
           tf.keras.metrics.Recall(name='recall'), 
           tf.keras.metrics.AUC(name='AUC')]

In [ ]:
# After creating full X and y arrays (could be 30K+)
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.uint8)

# ✅ Select first 5000 examples
X = X[:2000]
y = y[:2000]

# Then split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from tensorflow.keras import layers, models, Input

def build_unet(input_shape=(64, 64, 3)):
    inputs = Input(input_shape)

    # --- Encoder ---
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        return x

    def encoder_block(x, filters):
        f = conv_block(x, filters)
        p = layers.MaxPooling2D(pool_size=(2, 2))(f)
        return f, p

    def decoder_block(x, skip, filters):
        x = layers.Conv2DTranspose(filters, kernel_size=2, strides=2, padding='same')(x)
        x = layers.Concatenate()([x, skip])
        return conv_block(x, filters)

    # Downsampling path
    f1, p1 = encoder_block(inputs, 32)
    f2, p2 = encoder_block(p1, 64)
    f3, p3 = encoder_block(p2, 128)

    # Bottleneck
    b = conv_block(p3, 256)

    # Upsampling path
    d1 = decoder_block(b, f3, 128)
    d2 = decoder_block(d1, f2, 64)
    d3 = decoder_block(d2, f1, 32)

    # Output layer
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(d3)

    model = models.Model(inputs, outputs, name='UNet')
    return model

In [ ]:
model = build_unet((64, 64, 3))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()



In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=256,
    epochs=20,
    callbacks=[checkpoint_cb, early_stopping_cb, lr_scheduler_cb]
)

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(model, show_shapes=True, show_layer_names=True, to_file="unet_model.png")


In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pydicom




from tensorflow.keras.models import load_model

# --- Define DICOM path loader ---
def get_dicom_paths(directory):
    return [os.path.join(directory, fname) for fname in os.listdir(directory) if fname.endswith(".dcm")]

# --- Directories ---
TRAIN_IMG_DIR = r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_images"
train_img_path = get_dicom_paths(TRAIN_IMG_DIR)

# --- Load U-Net model ---
model = load_model(r'C:\Users\samya\PyCharmProject\Pneumonia-Detection\xray_model.h5')  # Replace with your actual model path

# --- Select a random image beyond the first 5000 ---
random_index = random.choice(range(5000, len(train_img_path)))
dicom_path = train_img_path[random_index]
print(f"Using DICOM file: {dicom_path}")

# --- Load and preprocess DICOM image ---
dicom = pydicom.dcmread(dicom_path)
image = dicom.pixel_array.astype(np.float32)
image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
original_size = image.shape

# Resize to model input size and convert to 3 channels
input_image = cv2.resize(image, (64, 64))
input_image = cv2.cvtColor(input_image, cv2.COLOR_GRAY2RGB)
input_image = input_image / 255.0
input_image = np.expand_dims(input_image, axis=0)  # Shape: (1, 64, 64, 3)

# --- Predict segmentation mask ---
predicted_mask = model.predict(input_image)[0, :, :, 0]
binary_mask = (predicted_mask > 0.5).astype(np.uint8) * 255
binary_mask_resized = cv2.resize(binary_mask, (original_size[1], original_size[0]))

# --- Find contours and draw bounding boxes ---
contours, _ = cv2.findContours(binary_mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
output_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(output_image, (x, y), (x + w, y + h), (0, 255, 0), 2)

# --- Display result ---
plt.figure(figsize=(10, 10))
plt.imshow(output_image)
plt.title(f"Detected Objects - Index {random_index}")
plt.axis('off')
plt.show()


In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pydicom
from tensorflow.keras.models import load_model

# --- Define DICOM path loader ---
def get_dicom_paths(directory):
    return [os.path.join(directory, fname) for fname in os.listdir(directory) if fname.endswith(".dcm")]

# --- Directories ---
TRAIN_IMG_DIR = r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_images"
TEST_IMG_DIR  = r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_test_images"
MODEL_PATH    = r'C:\Users\samya\PyCharmProject\Pneumonia-Detection\xray_model.h5'

# --- Load U-Net model ---
model = load_model(MODEL_PATH)

# --- Load paths ---
train_img_paths = get_dicom_paths(TRAIN_IMG_DIR)
test_img_paths = get_dicom_paths(TEST_IMG_DIR)

# --- Function to process and visualize with heatmap, predictions, and metadata ---
def process_and_display_with_boxes(image_paths, title):
    fig, axes = plt.subplots(3, 3, figsize=(14, 14))
    fig.suptitle(title, fontsize=18)

    for ax, path in zip(axes.flatten(), image_paths):
        ds = pydicom.dcmread(path)
        image = ds.pixel_array.astype(np.float32)
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        original_size = image.shape

        # Resize and prepare for model
        input_image = cv2.resize(image, (64, 64))
        input_image_rgb = cv2.cvtColor(input_image, cv2.COLOR_GRAY2RGB)
        input_image_norm = input_image_rgb / 255.0
        input_image_norm = np.expand_dims(input_image_norm, axis=0)

        # Predict and threshold
        predicted_mask = model.predict(input_image_norm)[0, :, :, 0]
        binary_mask = (predicted_mask > 0.5).astype(np.uint8) * 255
        binary_mask_resized = cv2.resize(binary_mask, (original_size[1], original_size[0]))

        # Apply heatmap to original grayscale image
        heatmap_image = cv2.applyColorMap(image, cv2.COLORMAP_VIRIDIS)


        # Find contours and draw bounding boxes
        contours, _ = cv2.findContours(binary_mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        output_image = heatmap_image.copy()
        box_labels = []

        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(output_image, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Green box
            box_labels.append(f"{w}x{h}")

        # --- Extract metadata ---
        patient_id = getattr(ds, "PatientID", "N/A")
        patient_age = getattr(ds, "PatientAge", "N/A")
        patient_sex = getattr(ds, "PatientSex", "N/A")
        dimensions = f"{original_size[0]}x{original_size[1]}"
        box_str = "; ".join(box_labels) if box_labels else "No box"

        # --- Show image ---
        ax.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(
            f"ID: {patient_id} | Age: {patient_age} | Sex: {patient_sex}\n"
            f"Img: {dimensions} | Box(es): {box_str}",
            fontsize=8
        )

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Leave room for suptitle
    plt.show()

# --- Randomly select 9 images from train and test ---
train_samples = random.sample(train_img_paths, 9)
test_samples = random.sample(test_img_paths, 9)

# --- Plot with heatmap, predictions, boxes, and metadata ---
process_and_display_with_boxes(train_samples, "Training DICOM Images with Bounding Boxes, Metadata & Heatmap")
process_and_display_with_boxes(test_samples, "Test DICOM Images with Bounding Boxes, Metadata & Heatmap")


In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pydicom
from tensorflow.keras.models import load_model

# --- Define DICOM path loader ---
def get_dicom_paths(directory):
    return [os.path.join(directory, fname) for fname in os.listdir(directory) if fname.endswith(".dcm")]

# --- Directories ---
TRAIN_IMG_DIR = r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_images"
TEST_IMG_DIR  = r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_test_images"
MODEL_PATH    = r'C:\Users\samya\PyCharmProject\Pneumonia-Detection\xray_model.h5'

# --- Load U-Net model ---
model = load_model(MODEL_PATH)

# --- Load paths ---
train_img_paths = get_dicom_paths(TRAIN_IMG_DIR)
test_img_paths = get_dicom_paths(TEST_IMG_DIR)

# --- Function to process and visualize with heatmap, predictions, and metadata ---
def process_and_display_with_boxes(image_paths, title):
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    fig.suptitle(title, fontsize=18)

    for ax, path in zip(axes.flatten(), image_paths):
        ds = pydicom.dcmread(path)
        image = ds.pixel_array.astype(np.float32)
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        original_size = image.shape

        # Resize and prepare for model
        input_image = cv2.resize(image, (64, 64))
        input_image_rgb = cv2.cvtColor(input_image, cv2.COLOR_GRAY2RGB)
        input_image_norm = input_image_rgb / 255.0
        input_image_norm = np.expand_dims(input_image_norm, axis=0)

        # Predict and threshold
        predicted_mask = model.predict(input_image_norm)[0, :, :, 0]
        binary_mask = (predicted_mask > 0.5).astype(np.uint8) * 255
        binary_mask_resized = cv2.resize(binary_mask, (original_size[1], original_size[0]))

        # Apply VIRIDIS colormap to original grayscale image
        heatmap_image = cv2.applyColorMap(image, cv2.COLORMAP_VIRIDIS)

        # Find contours and draw red bounding boxes
        contours, _ = cv2.findContours(binary_mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        output_image = heatmap_image.copy()
        box_labels = []

        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(output_image, (x, y), (x + w, y + h), (0, 0, 255), 2)  
            box_labels.append(f"{w}x{h}")

        # --- Extract metadata ---
        patient_id = getattr(ds, "PatientID", "N/A")
        patient_age = getattr(ds, "PatientAge", "N/A")
        patient_sex = getattr(ds, "PatientSex", "N/A")
        view_pos = getattr(ds, "ViewPosition", "N/A")
        dimensions = f"{original_size[0]}x{original_size[1]}"
        box_str = "; ".join(box_labels) if box_labels else "No box"

        # --- Show image ---
        ax.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(
            f"ID: {patient_id} | Age: {patient_age} | Sex: {patient_sex} | View: {view_pos}\n"
            f"Size: {dimensions} | Box(es): {box_str}",
            fontsize=8
        )

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Leave room for suptitle
    plt.show()

# --- Randomly select 9 images from train and test ---
train_samples = random.sample(train_img_paths, 9)
test_samples = random.sample(test_img_paths, 9)




In [ ]:
# --- Plot with heatmap, predictions, boxes, and metadata ---
process_and_display_with_boxes(train_samples, "Training DICOM Images with VIRIDIS Heatmap & Red Bounding Boxes")


In [ ]:
process_and_display_with_boxes(test_samples, "Test DICOM Images with VIRIDIS Heatmap & Red Bounding Boxes")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

def plot_predictions(
    model_path,
    image_dir,
    masks_dict,
    num_samples=6,
    img_size=64
):
    """
    Plot original image, ground truth mask, and predicted mask for a few samples.

    Args:
        model_path (str): Path to trained Keras model (.h5)
        image_dir (str): Directory containing .npy image files
        masks_dict (dict): Dictionary {patient_id: 2D mask array}
        num_samples (int): Number of samples to visualize
        img_size (int): Image/mask resolution (default: 64)
    """
    # Load model
    model = load_model(model_path)

    # Get .npy image files
    image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.npy')])
    image_files = image_files[:num_samples]  # limit to N samples

    # Create plot
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 3))

    for i, fname in enumerate(image_files):
        patient_id = os.path.splitext(fname)[0]
        image_path = os.path.join(image_dir, fname)

        # Load and preprocess image
        image = np.load(image_path) / 255.0
        if image.ndim == 2:
            image = image[..., np.newaxis]  # ensure shape (64, 64, 1)
        input_image = np.expand_dims(image, axis=0)  # shape: (1, 64, 64, 1)

        # Predict mask
        pred_mask = model.predict(input_image, verbose=0)[0, :, :, 0]
        pred_mask_bin = (pred_mask > 0.5).astype(np.uint8)

        # Load ground truth mask from dictionary
        gt_mask = masks_dict.get(patient_id, np.zeros((img_size, img_size)))

        # Plot image
        axes[i, 0].imshow(image.squeeze(), cmap='gray')
        axes[i, 0].set_title(f"Original Image\n{patient_id}")
        axes[i, 0].axis('off')

        # Plot ground truth mask
        axes[i, 1].imshow(gt_mask, cmap='gray')
        axes[i, 1].set_title("Ground Truth Mask")
        axes[i, 1].axis('off')

        # Plot predicted mask
        axes[i, 2].imshow(pred_mask_bin, cmap='gray')
        axes[i, 2].set_title("Predicted Mask")
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# Assumes you have: image_paths (list of .dcm files), pneumonia_labels (Target == 1 rows from CSV)

masks = {}
for path in image_paths:
    img_id = os.path.basename(path).replace('.dcm', '')
    dcm = pydicom.dcmread(path)
    orig_shape = dcm.pixel_array.shape  # Usually (1024, 1024) or similar
    mask = create_segmentation_mask(img_id, labels_df, orig_size=orig_shape, new_size=(64, 64))
    masks[img_id] = mask


In [ ]:
mask_pixels = [np.sum(m) for m in y_train]
print(np.mean(mask_pixels), np.max(mask_pixels))  # If all zeros, bad!



In [ ]:
# Filter for pneumonia-only samples
labels_df = pd.read_csv(r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_labels.csv")

pneumonia_ids = set(labels_df[labels_df["Target"] == 1]["patientId"])

# Keep only DICOMs with pneumonia
image_paths = [p for p in image_paths if os.path.basename(p).replace(".dcm", "") in pneumonia_ids]


In [ ]:
plt.imshow(pred_mask, cmap='hot')
plt.title("Raw Prediction Values")
plt.colorbar()
plt.show()


In [ ]:
plot_predictions(
    model_path=r"C:\Users\samya\PyCharmProject\Pneumonia-Detection\xray_model.h5",
    image_dir="npy_images",
    masks_dict=masks,
    num_samples=6  # or 10, 20, etc.
)


In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pydicom
from tensorflow.keras.models import load_model

# --- Define DICOM path loader ---
def get_dicom_paths(directory):
    return [os.path.join(directory, fname) for fname in os.listdir(directory) if fname.endswith(".dcm")]

# --- Directories ---
TRAIN_IMG_DIR = r"C:\Users\samya\PyCharmProject\Pneumonia-Detection_dataset\data\stage_2_train_images"
train_img_path = get_dicom_paths(TRAIN_IMG_DIR)

# --- Load U-Net model ---
model = load_model(r'C:\Users\samya\PyCharmProject\Pneumonia-Detection\xray_model.h5')  # Replace with your actual model path

# --- Select a random image beyond the first 5000 ---
random_index = random.choice(range(5000, len(train_img_path)))
dicom_path = train_img_path[random_index]
print(f"Using DICOM file: {dicom_path}")

# --- Load and preprocess DICOM image ---
dicom = pydicom.dcmread(dicom_path)
image = dicom.pixel_array.astype(np.float32)
image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
original_size = image.shape

# Resize to model input size and convert to 3 channels
input_image = cv2.resize(image, (64, 64))
input_image = cv2.cvtColor(input_image, cv2.COLOR_GRAY2RGB)
input_image = input_image / 255.0
input_image = np.expand_dims(input_image, axis=0)  # Shape: (1, 64, 64, 3)

# --- Predict segmentation mask ---
predicted_mask = model.predict(input_image)[0, :, :, 0]
binary_mask = (predicted_mask > 0.5).astype(np.uint8) * 255
binary_mask_resized = cv2.resize(binary_mask, (original_size[1], original_size[0]))

# --- Find contours and draw bounding boxes ---
contours, _ = cv2.findContours(binary_mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
output_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(output_image, (x, y), (x + w, y + h), (0, 255, 0), 2)

# --- Display result ---
plt.figure(figsize=(10, 10))
plt.imshow(output_image)
plt.title(f"Detected Objects - Index {random_index}")
plt.axis('off')
plt.show()